## Step 1: Initialize the 4n qubits

In [1]:
from qiskit import QuantumCircuit, QuantumRegister

from qiskit import transpile

# Qollab injects `backend` from the Select QPU dialog.
# Locally, fall back to Aer so the notebook still runs.
try:
    backend
except NameError:
    from qiskit_aer import AerSimulator
    backend = AerSimulator()


n = 5  # qubits per block; total = 4n = 20

a = QuantumRegister(n, "a")  # block 1: qubits 0-4
b = QuantumRegister(n, "b")  # block 2: qubits 5-9
c = QuantumRegister(n, "c")  # block 3: qubits 10-14
d = QuantumRegister(n, "d")  # block 4: qubits 15-19

qc = QuantumCircuit(a, b, c, d)
qc.draw()


a_0: 
     
a_1: 
     
a_2: 
     
a_3: 
     
a_4: 
     
b_0: 
     
b_1: 
     
b_2: 
     
b_3: 
     
b_4: 
     
c_0: 
     
c_1: 
     
c_2: 
     
c_3: 
     
c_4: 
     
d_0: 
     
d_1: 
     
d_2: 
     
d_3: 
     
d_4:

## Step 2: Generate physical Bell pairs between blocks b and c


In [2]:
# |Φ+> = (|00> + |11>)/√2 between b[i] and c[i]
for i in range(n):
    qc.h(b[i])
    qc.cx(b[i], c[i])

qc.barrier()
qc.draw()


░ 
a_0: ───────────────────────────────░─
                                    ░ 
a_1: ───────────────────────────────░─
                                    ░ 
a_2: ───────────────────────────────░─
                                    ░ 
a_3: ───────────────────────────────░─
                                    ░ 
a_4: ───────────────────────────────░─
     ┌───┐                          ░ 
b_0: ┤ H ├──■───────────────────────░─
     ├───┤  │                       ░ 
b_1: ┤ H ├──┼────■──────────────────░─
     ├───┤  │    │                  ░ 
b_2: ┤ H ├──┼────┼────■─────────────░─
     ├───┤  │    │    │             ░ 
b_3: ┤ H ├──┼────┼────┼────■────────░─
     ├───┤  │    │    │    │        ░ 
b_4: ┤ H ├──┼────┼────┼────┼────■───░─
     └───┘┌─┴─┐  │    │    │    │   ░ 
c_0: ─────┤ X ├──┼────┼────┼────┼───░─
          └───┘┌─┴─┐  │    │    │   ░ 
c_1: ──────────┤ X ├──┼────┼────┼───░─
               └───┘┌─┴─┐  │    │   ░ 
c_2: ───────────────┤ X ├──┼────┼───░─
                    └───┘┌─┴─┐  │   ░ 
c_3: ────────────────────┤ X ├──┼───░─
                         └───┘┌─┴─┐ ░ 
c_4: ─────────────────────────┤ X ├─░─
                              └───┘ ░ 
d_0: ───────────────────────────────░─
                                    ░ 
d_1: ───────────────────────────────░─
                                    ░ 
d_2: ───────────────────────────────░─
                                    ░ 
d_3: ───────────────────────────────░─
                                    ░ 
d_4: ───────────────────────────────░─
                                    ░

## Step 3: Encode blocks a and d with the 5-qubit repetition code

In [3]:
qc.h(a[0])  # a starts in |+> so the link becomes an encoded Bell pair

# α|0> + β|1> -> α|00000> + β|11111>
for block in (a, d):
    for k in range(1, n):
        qc.cx(block[0], block[k])

qc.barrier()
qc.draw()


░ ┌───┐                     ░ 
a_0: ───────────────────────────────░─┤ H ├──■────■────■────■───░─
                                    ░ └───┘┌─┴─┐  │    │    │   ░ 
a_1: ───────────────────────────────░──────┤ X ├──┼────┼────┼───░─
                                    ░      └───┘┌─┴─┐  │    │   ░ 
a_2: ───────────────────────────────░───────────┤ X ├──┼────┼───░─
                                    ░           └───┘┌─┴─┐  │   ░ 
a_3: ───────────────────────────────░────────────────┤ X ├──┼───░─
                                    ░                └───┘┌─┴─┐ ░ 
a_4: ───────────────────────────────░─────────────────────┤ X ├─░─
     ┌───┐                          ░                     └───┘ ░ 
b_0: ┤ H ├──■───────────────────────░───────────────────────────░─
     ├───┤  │                       ░                           ░ 
b_1: ┤ H ├──┼────■──────────────────░───────────────────────────░─
     ├───┤  │    │                  ░                           ░ 
b_2: ┤ H ├──┼────┼────■─────────────░───────────────────────────░─
     ├───┤  │    │    │             ░                           ░ 
b_3: ┤ H ├──┼────┼────┼────■────────░───────────────────────────░─
     ├───┤  │    │    │    │        ░                           ░ 
b_4: ┤ H ├──┼────┼────┼────┼────■───░───────────────────────────░─
     └───┘┌─┴─┐  │    │    │    │   ░                           ░ 
c_0: ─────┤ X ├──┼────┼────┼────┼───░───────────────────────────░─
          └───┘┌─┴─┐  │    │    │   ░                           ░ 
c_1: ──────────┤ X ├──┼────┼────┼───░───────────────────────────░─
               └───┘┌─┴─┐  │    │   ░                           ░ 
c_2: ───────────────┤ X ├──┼────┼───░───────────────────────────░─
                    └───┘┌─┴─┐  │   ░                           ░ 
c_3: ────────────────────┤ X ├──┼───░───────────────────────────░─
                         └───┘┌─┴─┐ ░                           ░ 
c_4: ─────────────────────────┤ X ├─░───────────────────────────░─
                              └───┘ ░                           ░ 
d_0: ───────────────────────────────░───■────■────■────■────────░─
                                    ░ ┌─┴─┐  │    │    │        ░ 
d_1: ───────────────────────────────░─┤ X ├──┼────┼────┼────────░─
                                    ░ └───┘┌─┴─┐  │    │        ░ 
d_2: ───────────────────────────────░──────┤ X ├──┼────┼────────░─
                                    ░      └───┘┌─┴─┐  │        ░ 
d_3: ───────────────────────────────░───────────┤ X ├──┼────────░─
                                    ░           └───┘┌─┴─┐      ░ 
d_4: ───────────────────────────────░────────────────┤ X ├──────░─
                                    ░                └───┘      ░

## Step 4: Teleported CNOT from block a to block d, using the b–c Bell pairs

In [4]:
for i in range(n):
    qc.cx(a[i], b[i])   # control's side: CNOT onto its half of the pair
    qc.cx(c[i], d[i])   # replaces "measure b in Z, apply X to c"
    qc.cx(b[i], d[i])
    qc.h(c[i])
    qc.cz(c[i], a[i])   # replaces "measure c in X, apply Z to a"
    qc.barrier()

qc.draw()


░ ┌───┐                     ░           »
a_0: ───────────────────────────────░─┤ H ├──■────■────■────■───░───■───────»
                                    ░ └───┘┌─┴─┐  │    │    │   ░   │       »
a_1: ───────────────────────────────░──────┤ X ├──┼────┼────┼───░───┼───────»
                                    ░      └───┘┌─┴─┐  │    │   ░   │       »
a_2: ───────────────────────────────░───────────┤ X ├──┼────┼───░───┼───────»
                                    ░           └───┘┌─┴─┐  │   ░   │       »
a_3: ───────────────────────────────░────────────────┤ X ├──┼───░───┼───────»
                                    ░                └───┘┌─┴─┐ ░   │       »
a_4: ───────────────────────────────░─────────────────────┤ X ├─░───┼───────»
     ┌───┐                          ░                     └───┘ ░ ┌─┴─┐     »
b_0: ┤ H ├──■───────────────────────░───────────────────────────░─┤ X ├──■──»
     ├───┤  │                       ░                           ░ └───┘  │  »
b_1: ┤ H ├──┼────■──────────────────░───────────────────────────░────────┼──»
     ├───┤  │    │                  ░                           ░        │  »
b_2: ┤ H ├──┼────┼────■─────────────░───────────────────────────░────────┼──»
     ├───┤  │    │    │             ░                           ░        │  »
b_3: ┤ H ├──┼────┼────┼────■────────░───────────────────────────░────────┼──»
     ├───┤  │    │    │    │        ░                           ░        │  »
b_4: ┤ H ├──┼────┼────┼────┼────■───░───────────────────────────░────────┼──»
     └───┘┌─┴─┐  │    │    │    │   ░                           ░        │  »
c_0: ─────┤ X ├──┼────┼────┼────┼───░───────────────────────────░───■────┼──»
          └───┘┌─┴─┐  │    │    │   ░                           ░   │    │  »
c_1: ──────────┤ X ├──┼────┼────┼───░───────────────────────────░───┼────┼──»
               └───┘┌─┴─┐  │    │   ░                           ░   │    │  »
c_2: ───────────────┤ X ├──┼────┼───░───────────────────────────░───┼────┼──»
                    └───┘┌─┴─┐  │   ░                           ░   │    │  »
c_3: ────────────────────┤ X ├──┼───░───────────────────────────░───┼────┼──»
                         └───┘┌─┴─┐ ░                           ░   │    │  »
c_4: ─────────────────────────┤ X ├─░───────────────────────────░───┼────┼──»
                              └───┘ ░                           ░ ┌─┴─┐┌─┴─┐»
d_0: ───────────────────────────────░───■────■────■────■────────░─┤ X ├┤ X ├»
                                    ░ ┌─┴─┐  │    │    │        ░ └───┘└───┘»
d_1: ───────────────────────────────░─┤ X ├──┼────┼────┼────────░───────────»
                                    ░ └───┘┌─┴─┐  │    │        ░           »
d_2: ───────────────────────────────░──────┤ X ├──┼────┼────────░───────────»
                                    ░      └───┘┌─┴─┐  │        ░           »
d_3: ───────────────────────────────░───────────┤ X ├──┼────────░───────────»
                                    ░           └───┘┌─┴─┐      ░           »
d_4: ───────────────────────────────░────────────────┤ X ├──────░───────────»
                                    ░                └───┘      ░           »
«              ░                    ░                    ░                   »
«a_0: ──────■──░────────────────────░────────────────────░───────────────────»
«           │  ░                    ░                    ░                   »
«a_1: ──────┼──░───■─────────────■──░────────────────────░───────────────────»
«           │  ░   │             │  ░                    ░                   »
«a_2: ──────┼──░───┼─────────────┼──░───■─────────────■──░───────────────────»
«           │  ░   │             │  ░   │             │  ░                   »
«a_3: ──────┼──░───┼─────────────┼──░───┼─────────────┼──░───■─────────────■─»
«           │  ░   │             │  ░   │             │  ░   │             │ »
«a_4: ──────┼──░───┼─────────────┼──░───┼─────────────┼──░───┼─────────────┼─»
«           │  ░   │             │  ░   │       

## Step 5: Remove blocks b and c

In [5]:
from qiskit.quantum_info import Statevector, partial_trace, state_fidelity

# Equivalent circuit on blocks a and d only
qc_ad = QuantumCircuit(a, d)

qc_ad.h(a[0])                   # Step 3: |+> on a
for block in (a, d):            # Step 3: repetition-code encoding
    for k in range(1, n):
        qc_ad.cx(block[0], block[k])
qc_ad.barrier()

for i in range(n):              # Step 4: what the teleported CNOT did
    qc_ad.cx(a[i], d[i])
    qc_ad.barrier()

# Check: trace out b and c from the full circuit and compare
bc = [qc.find_bit(q).index for q in (*b, *c)]
rho_ad = partial_trace(Statevector(qc), bc)
print("purity:  ", rho_ad.purity().real)                            # 1.0 → b, c fully disentangled
print("fidelity:", state_fidelity(rho_ad, Statevector(qc_ad)))      # 1.0 → same state on a, d

qc_ad.draw()


purity:   0.9999999999999998
fidelity: 0.9999999999999997


┌───┐                     ░       ░       ░       ░       ░       ░ 
a_0: ┤ H ├──■────■────■────■───░───■───░───────░───────░───────░───────░─
     └───┘┌─┴─┐  │    │    │   ░   │   ░       ░       ░       ░       ░ 
a_1: ─────┤ X ├──┼────┼────┼───░───┼───░───■───░───────░───────░───────░─
          └───┘┌─┴─┐  │    │   ░   │   ░   │   ░       ░       ░       ░ 
a_2: ──────────┤ X ├──┼────┼───░───┼───░───┼───░───■───░───────░───────░─
               └───┘┌─┴─┐  │   ░   │   ░   │   ░   │   ░       ░       ░ 
a_3: ───────────────┤ X ├──┼───░───┼───░───┼───░───┼───░───■───░───────░─
                    └───┘┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░       ░ 
a_4: ────────────────────┤ X ├─░───┼───░───┼───░───┼───░───┼───░───■───░─
                         └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░ 
d_0: ──■────■────■────■────────░─┤ X ├─░───┼───░───┼───░───┼───░───┼───░─
     ┌─┴─┐  │    │    │        ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░ 
d_1: ┤ X ├──┼────┼────┼────────░───────░─┤ X ├─░───┼───░───┼───░───┼───░─
     └───┘┌─┴─┐  │    │        ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░ 
d_2: ─────┤ X ├──┼────┼────────░───────░───────░─┤ X ├─░───┼───░───┼───░─
          └───┘┌─┴─┐  │        ░       ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░ 
d_3: ──────────┤ X ├──┼────────░───────░───────░───────░─┤ X ├─░───┼───░─
               └───┘┌─┴─┐      ░       ░       ░       ░ └───┘ ░ ┌─┴─┐ ░ 
d_4: ───────────────┤ X ├──────░───────░───────░───────░───────░─┤ X ├─░─
                    └───┘      ░       ░       ░       ░       ░ └───┘ ░

## Step 6: Repeat Steps 1–5 for a second link

In [6]:
# Step 1: initialize
a2 = QuantumRegister(n, "a2")
b2 = QuantumRegister(n, "b2")
c2 = QuantumRegister(n, "c2")
d2 = QuantumRegister(n, "d2")

qc2 = QuantumCircuit(a2, b2, c2, d2)

# Step 2: physical Bell pairs between b2[i] and c2[i]
for i in range(n):
    qc2.h(b2[i])
    qc2.cx(b2[i], c2[i])
qc2.barrier()

# Step 3: repetition-code encoding on a2 and d2 (a2 starts in |+>)
qc2.h(a2[0])
for block in (a2, d2):
    for k in range(1, n):
        qc2.cx(block[0], block[k])
qc2.barrier()

# Step 4: teleported CNOT a2 -> d2 using the b2–c2 Bell pairs
for i in range(n):
    qc2.cx(a2[i], b2[i])
    qc2.cx(c2[i], d2[i])
    qc2.cx(b2[i], d2[i])
    qc2.h(c2[i])
    qc2.cz(c2[i], a2[i])
    qc2.barrier()

# Step 5: equivalent circuit on a2 and d2 only
qc_ad2 = QuantumCircuit(a2, d2)

qc_ad2.h(a2[0])
for block in (a2, d2):
    for k in range(1, n):
        qc_ad2.cx(block[0], block[k])
qc_ad2.barrier()

for i in range(n):
    qc_ad2.cx(a2[i], d2[i])
    qc_ad2.barrier()

bc2 = [qc2.find_bit(q).index for q in (*b2, *c2)]
rho_ad2 = partial_trace(Statevector(qc2), bc2)
print("purity:  ", rho_ad2.purity().real)
print("fidelity:", state_fidelity(rho_ad2, Statevector(qc_ad2)))

qc_ad2.draw()


purity:   0.9999999999999998
fidelity: 0.9999999999999997


┌───┐                     ░       ░       ░       ░       ░       ░ 
a2_0: ┤ H ├──■────■────■────■───░───■───░───────░───────░───────░───────░─
      └───┘┌─┴─┐  │    │    │   ░   │   ░       ░       ░       ░       ░ 
a2_1: ─────┤ X ├──┼────┼────┼───░───┼───░───■───░───────░───────░───────░─
           └───┘┌─┴─┐  │    │   ░   │   ░   │   ░       ░       ░       ░ 
a2_2: ──────────┤ X ├──┼────┼───░───┼───░───┼───░───■───░───────░───────░─
                └───┘┌─┴─┐  │   ░   │   ░   │   ░   │   ░       ░       ░ 
a2_3: ───────────────┤ X ├──┼───░───┼───░───┼───░───┼───░───■───░───────░─
                     └───┘┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░       ░ 
a2_4: ────────────────────┤ X ├─░───┼───░───┼───░───┼───░───┼───░───■───░─
                          └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░ 
d2_0: ──■────■────■────■────────░─┤ X ├─░───┼───░───┼───░───┼───░───┼───░─
      ┌─┴─┐  │    │    │        ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░ 
d2_1: ┤ X ├──┼────┼────┼────────░───────░─┤ X ├─░───┼───░───┼───░───┼───░─
      └───┘┌─┴─┐  │    │        ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░ 
d2_2: ─────┤ X ├──┼────┼────────░───────░───────░─┤ X ├─░───┼───░───┼───░─
           └───┘┌─┴─┐  │        ░       ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░ 
d2_3: ──────────┤ X ├──┼────────░───────░───────░───────░─┤ X ├─░───┼───░─
                └───┘┌─┴─┐      ░       ░       ░       ░ └───┘ ░ ┌─┴─┐ ░ 
d2_4: ───────────────┤ X ├──────░───────░───────░───────░───────░─┤ X ├─░─
                     └───┘      ░       ░       ░       ░       ░ └───┘ ░

## Step 7: Combine the two links

In [7]:
# Combined chain: link 1 (a–d) and link 2 (a2–d2), 4n = 20 qubits
qc_chain = QuantumCircuit(a, d, a2, d2)
qc_chain.compose(qc_ad, qubits=[*a, *d], inplace=True)
qc_chain.compose(qc_ad2, qubits=[*a2, *d2], inplace=True)

qc_chain.draw()


┌───┐                     ░       ░       ░       ░       ░       ░ 
 a_0: ┤ H ├──■────■────■────■───░───■───░───────░───────░───────░───────░─
      └───┘┌─┴─┐  │    │    │   ░   │   ░       ░       ░       ░       ░ 
 a_1: ─────┤ X ├──┼────┼────┼───░───┼───░───■───░───────░───────░───────░─
           └───┘┌─┴─┐  │    │   ░   │   ░   │   ░       ░       ░       ░ 
 a_2: ──────────┤ X ├──┼────┼───░───┼───░───┼───░───■───░───────░───────░─
                └───┘┌─┴─┐  │   ░   │   ░   │   ░   │   ░       ░       ░ 
 a_3: ───────────────┤ X ├──┼───░───┼───░───┼───░───┼───░───■───░───────░─
                     └───┘┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░       ░ 
 a_4: ────────────────────┤ X ├─░───┼───░───┼───░───┼───░───┼───░───■───░─
                          └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░ 
 d_0: ──■────■────■────■────────░─┤ X ├─░───┼───░───┼───░───┼───░───┼───░─
      ┌─┴─┐  │    │    │        ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░ 
 d_1: ┤ X ├──┼────┼────┼────────░───────░─┤ X ├─░───┼───░───┼───░───┼───░─
      └───┘┌─┴─┐  │    │        ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░ 
 d_2: ─────┤ X ├──┼────┼────────░───────░───────░─┤ X ├─░───┼───░───┼───░─
           └───┘┌─┴─┐  │        ░       ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░ 
 d_3: ──────────┤ X ├──┼────────░───────░───────░───────░─┤ X ├─░───┼───░─
                └───┘┌─┴─┐      ░       ░       ░       ░ └───┘ ░ ┌─┴─┐ ░ 
 d_4: ───────────────┤ X ├──────░───────░───────░───────░───────░─┤ X ├─░─
      ┌───┐          └───┘      ░       ░       ░       ░       ░ └───┘ ░ 
a2_0: ┤ H ├──■────■────■────■───░───■───░───────░───────░───────░───────░─
      └───┘┌─┴─┐  │    │    │   ░   │   ░       ░       ░       ░       ░ 
a2_1: ─────┤ X ├──┼────┼────┼───░───┼───░───■───░───────░───────░───────░─
           └───┘┌─┴─┐  │    │   ░   │   ░   │   ░       ░       ░       ░ 
a2_2: ──────────┤ X ├──┼────┼───░───┼───░───┼───░───■───░───────░───────░─
                └───┘┌─┴─┐  │   ░   │   ░   │   ░   │   ░       ░       ░ 
a2_3: ───────────────┤ X ├──┼───░───┼───░───┼───░───┼───░───■───░───────░─
                     └───┘┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░       ░ 
a2_4: ────────────────────┤ X ├─░───┼───░───┼───░───┼───░───┼───░───■───░─
                          └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░ 
d2_0: ──■────■────■────■────────░─┤ X ├─░───┼───░───┼───░───┼───░───┼───░─
      ┌─┴─┐  │    │    │        ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░ 
d2_1: ┤ X ├──┼────┼────┼────────░───────░─┤ X ├─░───┼───░───┼───░───┼───░─
      └───┘┌─┴─┐  │    │        ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░ 
d2_2: ─────┤ X ├──┼────┼────────░───────░───────░─┤ X ├─░───┼───░───┼───░─
           └───┘┌─┴─┐  │        ░       ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░ 
d2_3: ──────────┤ X ├──┼────────░───────░───────░───────░─┤ X ├─░───┼───░─
                └───┘┌─┴─┐      ░       ░       ░       ░ └───┘ ░ ┌─┴─┐ ░ 
d2_4: ───────────────┤ X ├──────░───────░───────░───────░───────░─┤ X ├─░─
                     └───┘      ░       ░       ░       ░       ░ └───┘ ░

## Step 8: Entanglement swap via logical BSM at the middle node (d, a2)

In [8]:
from qiskit import ClassicalRegister

mx = ClassicalRegister(n, "mx")  # X-basis results from d
mz = ClassicalRegister(n, "mz")  # Z-basis results from a2
qc_chain.add_register(mx, mz)

for i in range(n):
    qc_chain.cx(d[i], a2[i])
qc_chain.barrier()

qc_chain.h(d)              # X-basis measurement = H, then measure
qc_chain.measure(d, mx)
qc_chain.measure(a2, mz)

qc_chain.draw()


┌───┐                     ░       ░       ░       ░       ░       ░      »
 a_0: ┤ H ├──■────■────■────■───░───■───░───────░───────░───────░───────░──────»
      └───┘┌─┴─┐  │    │    │   ░   │   ░       ░       ░       ░       ░      »
 a_1: ─────┤ X ├──┼────┼────┼───░───┼───░───■───░───────░───────░───────░──────»
           └───┘┌─┴─┐  │    │   ░   │   ░   │   ░       ░       ░       ░      »
 a_2: ──────────┤ X ├──┼────┼───░───┼───░───┼───░───■───░───────░───────░──────»
                └───┘┌─┴─┐  │   ░   │   ░   │   ░   │   ░       ░       ░      »
 a_3: ───────────────┤ X ├──┼───░───┼───░───┼───░───┼───░───■───░───────░──────»
                     └───┘┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░       ░      »
 a_4: ────────────────────┤ X ├─░───┼───░───┼───░───┼───░───┼───░───■───░──────»
                          └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░      »
 d_0: ──■────■────■────■────────░─┤ X ├─░───┼───░───┼───░───┼───░───┼───░───■──»
      ┌─┴─┐  │    │    │        ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░   │  »
 d_1: ┤ X ├──┼────┼────┼────────░───────░─┤ X ├─░───┼───░───┼───░───┼───░───┼──»
      └───┘┌─┴─┐  │    │        ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │  »
 d_2: ─────┤ X ├──┼────┼────────░───────░───────░─┤ X ├─░───┼───░───┼───░───┼──»
           └───┘┌─┴─┐  │        ░       ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │  »
 d_3: ──────────┤ X ├──┼────────░───────░───────░───────░─┤ X ├─░───┼───░───┼──»
                └───┘┌─┴─┐      ░       ░       ░       ░ └───┘ ░ ┌─┴─┐ ░   │  »
 d_4: ───────────────┤ X ├──────░───────░───────░───────░───────░─┤ X ├─░───┼──»
      ┌───┐          └───┘      ░       ░       ░       ░       ░ └───┘ ░ ┌─┴─┐»
a2_0: ┤ H ├──■────■────■────■───░───■───░───────░───────░───────░───────░─┤ X ├»
      └───┘┌─┴─┐  │    │    │   ░   │   ░       ░       ░       ░       ░ └───┘»
a2_1: ─────┤ X ├──┼────┼────┼───░───┼───░───■───░───────░───────░───────░──────»
           └───┘┌─┴─┐  │    │   ░   │   ░   │   ░       ░       ░       ░      »
a2_2: ──────────┤ X ├──┼────┼───░───┼───░───┼───░───■───░───────░───────░──────»
                └───┘┌─┴─┐  │   ░   │   ░   │   ░   │   ░       ░       ░      »
a2_3: ───────────────┤ X ├──┼───░───┼───░───┼───░───┼───░───■───░───────░──────»
                     └───┘┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░       ░      »
a2_4: ────────────────────┤ X ├─░───┼───░───┼───░───┼───░───┼───░───■───░──────»
                          └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░      »
d2_0: ──■────■────■────■────────░─┤ X ├─░───┼───░───┼───░───┼───░───┼───░──────»
      ┌─┴─┐  │    │    │        ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░      »
d2_1: ┤ X ├──┼────┼────┼────────░───────░─┤ X ├─░───┼───░───┼───░───┼───░──────»
      └───┘┌─┴─┐  │    │        ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░      »
d2_2: ─────┤ X ├──┼────┼────────░───────░───────░─┤ X ├─░───┼───░───┼───░──────»
           └───┘┌─┴─┐  │        ░       ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░      »
d2_3: ──────────┤ X ├──┼────────░───────░───────░───────░─┤ X ├─░───┼───░──────»
                └───┘┌─┴─┐      ░       ░       ░       ░ └───┘ ░ ┌─┴─┐ ░      »
d2_4: ───────────────┤ X ├──────░───────░───────░───────░───────░─┤ X ├─░──────»
                     └───┘      ░       ░       ░       ░       ░ └───┘ ░      »
mx: 5/═════════════════════════════════════════════════════════════════════════»
                                                                               »
mz: 5/═════════════════════════════════════════════════════════════════════════»
                                                                               »
«                           ░                                 
« a_0: ─────────────────────░─────────────────────────────────
«                           ░                                 
« a_1: ─────────────────────░─────────────────────────────────
«                           ░                                 
« a_2: ─────────────────────░─────────────────

## Step 9: Classical error correction, then the logical BSM bits

In [9]:
from collections import Counter

def correct_repetition(bits):
    """Decode an n-bit repetition codeword by majority vote (fixes up to (n-1)//2 = 2 bit flips)."""
    majority = int(sum(bits) > len(bits) // 2)
    return [majority] * len(bits)

def xor_all(bits):
    out = 0
    for bit in bits:
        out ^= bit
    return out

result = backend.run(transpile(qc_chain, backend), shots=1000).result()
counts = result.get_counts()

frames = Counter()
for key, cnt in counts.items():
    mz_str, mx_str = key.split()                 # Qiskit prints 'mz mx'
    mx_bits = [int(x) for x in mx_str[::-1]]     # reverse: bit 0 is rightmost
    mz_bits = [int(x) for x in mz_str[::-1]]

    m_x = xor_all(mx_bits)                       # no correction possible
    m_z = xor_all(correct_repetition(mz_bits))   # correct, then XOR

    frames[(m_x, m_z)] += cnt

print(frames)   # (m_x, m_z) → apply Z_L^{m_x} X_L^{m_z} to d2


Counter({(0, 1): 263, (0, 0): 263, (1, 1): 260, (1, 0): 214})


## Step 10: Pauli frame corrections

In [10]:
from functools import reduce
from itertools import combinations

from qiskit.circuit.classical import expr

# Logical BSM bits, computed in-circuit from the Step 8 results
m_x_expr = reduce(expr.bit_xor, mx)                    # parity of d (X basis)

# majority of a2 (Z basis): true if any n//2 + 1 bits are all 1
m_z_expr = reduce(expr.logic_or, [
    reduce(expr.logic_and, group) for group in combinations(mz, n // 2 + 1)
])

# Pauli frame corrections on d2: X_L^{m_z} Z_L^{m_x}
with qc_chain.if_test(m_z_expr):
    qc_chain.x(d2)          # X_L = XXXXX
with qc_chain.if_test(m_x_expr):
    qc_chain.z(d2[0])       # Z_L = Z on one qubit
qc_chain.barrier()

result = backend.run(transpile(qc_chain, backend), shots=1000).result()
counts = result.get_counts()

frames = Counter()
for key, cnt in counts.items():
    mz_str, mx_str = key.split()
    m_x = xor_all([int(x) for x in mx_str[::-1]])
    m_z = xor_all(correct_repetition([int(x) for x in mz_str[::-1]]))
    frames[(m_x, m_z)] += cnt

print(frames)   # each shot was corrected with its own (m_x, m_z)

qc_chain.draw()


Counter({(1, 0): 280, (1, 1): 244, (0, 0): 242, (0, 1): 234})


┌───┐                     ░       ░       ░       ░       ░       ░      »
 a_0: ┤ H ├──■────■────■────■───░───■───░───────░───────░───────░───────░──────»
      └───┘┌─┴─┐  │    │    │   ░   │   ░       ░       ░       ░       ░      »
 a_1: ─────┤ X ├──┼────┼────┼───░───┼───░───■───░───────░───────░───────░──────»
           └───┘┌─┴─┐  │    │   ░   │   ░   │   ░       ░       ░       ░      »
 a_2: ──────────┤ X ├──┼────┼───░───┼───░───┼───░───■───░───────░───────░──────»
                └───┘┌─┴─┐  │   ░   │   ░   │   ░   │   ░       ░       ░      »
 a_3: ───────────────┤ X ├──┼───░───┼───░───┼───░───┼───░───■───░───────░──────»
                     └───┘┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░       ░      »
 a_4: ────────────────────┤ X ├─░───┼───░───┼───░───┼───░───┼───░───■───░──────»
                          └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░      »
 d_0: ──■────■────■────■────────░─┤ X ├─░───┼───░───┼───░───┼───░───┼───░───■──»
      ┌─┴─┐  │    │    │        ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░   │  »
 d_1: ┤ X ├──┼────┼────┼────────░───────░─┤ X ├─░───┼───░───┼───░───┼───░───┼──»
      └───┘┌─┴─┐  │    │        ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │  »
 d_2: ─────┤ X ├──┼────┼────────░───────░───────░─┤ X ├─░───┼───░───┼───░───┼──»
           └───┘┌─┴─┐  │        ░       ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │  »
 d_3: ──────────┤ X ├──┼────────░───────░───────░───────░─┤ X ├─░───┼───░───┼──»
                └───┘┌─┴─┐      ░       ░       ░       ░ └───┘ ░ ┌─┴─┐ ░   │  »
 d_4: ───────────────┤ X ├──────░───────░───────░───────░───────░─┤ X ├─░───┼──»
      ┌───┐          └───┘      ░       ░       ░       ░       ░ └───┘ ░ ┌─┴─┐»
a2_0: ┤ H ├──■────■────■────■───░───■───░───────░───────░───────░───────░─┤ X ├»
      └───┘┌─┴─┐  │    │    │   ░   │   ░       ░       ░       ░       ░ └───┘»
a2_1: ─────┤ X ├──┼────┼────┼───░───┼───░───■───░───────░───────░───────░──────»
           └───┘┌─┴─┐  │    │   ░   │   ░   │   ░       ░       ░       ░      »
a2_2: ──────────┤ X ├──┼────┼───░───┼───░───┼───░───■───░───────░───────░──────»
                └───┘┌─┴─┐  │   ░   │   ░   │   ░   │   ░       ░       ░      »
a2_3: ───────────────┤ X ├──┼───░───┼───░───┼───░───┼───░───■───░───────░──────»
                     └───┘┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░       ░      »
a2_4: ────────────────────┤ X ├─░───┼───░───┼───░───┼───░───┼───░───■───░──────»
                          └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░   │   ░      »
d2_0: ──■────■────■────■────────░─┤ X ├─░───┼───░───┼───░───┼───░───┼───░──────»
      ┌─┴─┐  │    │    │        ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░   │   ░      »
d2_1: ┤ X ├──┼────┼────┼────────░───────░─┤ X ├─░───┼───░───┼───░───┼───░──────»
      └───┘┌─┴─┐  │    │        ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░   │   ░      »
d2_2: ─────┤ X ├──┼────┼────────░───────░───────░─┤ X ├─░───┼───░───┼───░──────»
           └───┘┌─┴─┐  │        ░       ░       ░ └───┘ ░ ┌─┴─┐ ░   │   ░      »
d2_3: ──────────┤ X ├──┼────────░───────░───────░───────░─┤ X ├─░───┼───░──────»
                └───┘┌─┴─┐      ░       ░       ░       ░ └───┘ ░ ┌─┴─┐ ░      »
d2_4: ───────────────┤ X ├──────░───────░───────░───────░───────░─┤ X ├─░──────»
                     └───┘      ░       ░       ░       ░       ░ └───┘ ░      »
mx: 5/═════════════════════════════════════════════════════════════════════════»
                                                                               »
mz: 5/═════════════════════════════════════════════════════════════════════════»
                                                                               »
«                           ░                                 »
« a_0: ─────────────────────░─────────────────────────────────»
«                           ░                                 »
« a_1: ─────────────────────░─────────────────────────────────»
«                           ░                                 »
« a_2: ─────────────────────░────────────

## Step 11: Bell pair fidelity

In [11]:
def logical_correlator(basis, shots=1000):
    """Estimate <P_L ⊗ P_L> on blocks a and d2 for P = X, Y or Z."""
    qc_m = qc_chain.copy()
    ea = ClassicalRegister(n, "ea")
    ed = ClassicalRegister(n, "ed")
    qc_m.add_register(ea, ed)
    for block in (a, d2):
        if basis == "X":            # X_L = X X X X X
            qc_m.h(block)
        elif basis == "Y":          # Y_L = Y X X X X
            qc_m.sdg(block[0])
            qc_m.h(block)
    qc_m.measure(a, ea)
    qc_m.measure(d2, ed)

    counts = backend.run(transpile(qc_m, backend), shots=shots).result().get_counts()
    total = 0
    for key, cnt in counts.items():
        ed_str, ea_str = key.split()[:2]           # Qiskit prints 'ed ea mz mx'
        ea_bits = [int(x) for x in ea_str[::-1]]
        ed_bits = [int(x) for x in ed_str[::-1]]
        if basis == "Z":    # decode each block first (classical error correction)
            parity = xor_all(correct_repetition(ea_bits)) ^ xor_all(correct_repetition(ed_bits))
        else:               # X_L and Y_L are read from the parity of all n qubits
            parity = xor_all(ea_bits + ed_bits)
        total += cnt * (1 - 2 * parity)            # +1 for even, -1 for odd
    return total / shots

XX = logical_correlator("X")
YY = logical_correlator("Y")
ZZ = logical_correlator("Z")

# Fidelity with |Φ+_L> = (|0_L 0_L> + |1_L 1_L>)/√2
F = (1 + XX - YY + ZZ) / 4
print(f"<XX> = {XX:+.3f}, <YY> = {YY:+.3f}, <ZZ> = {ZZ:+.3f}")
print(f"Bell pair fidelity F = {F:.3f}")

<XX> = +1.000, <YY> = -1.000, <ZZ> = +1.000
Bell pair fidelity F = 1.000
